# **Hands-on Lab: Interactive Visual Analytics with Folium**

The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site. In this lab you perform interactive visual analytics with `Folium`.

## Objectives
* **TASK 1:** Mark all launch sites on a map
* **TASK 2:** Mark the success/failed launches for each site on the map
* **TASK 3:** Calculate the distances between a launch site to its proximities

## Setup

In [1]:
# Run once in a terminal or notebook if the packages are missing:
# %pip install folium pandas
import folium
import pandas as pd

In [2]:
# Folium plugins
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon

## Load the data
The original lab used `js.fetch` (JupyterLite only). Locally, `pd.read_csv` can read the URL directly.

In [3]:
URL = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv'
spacex_df = pd.read_csv(URL)
spacex_df.head()

,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


## Task 1: Mark all launch sites on a map

In [4]:
# Keep the relevant columns and get one row per launch site
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [5]:
# Start location is NASA Johnson Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

# Circle with a popup label
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(
    folium.Popup('NASA Johnson Space Center')
)
# Text label marker
marker = folium.map.Marker(
    nasa_coordinate,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
    )
)
site_map.add_child(circle)
site_map.add_child(marker)
site_map

Now add a circle and a label for **each launch site**.

In [6]:
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for index, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    circle = folium.Circle(coordinate, radius=1000, color='#d35400', fill=True).add_child(
        folium.Popup(row['Launch Site'])
    )
    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' % row['Launch Site'],
        )
    )
    site_map.add_child(circle)
    site_map.add_child(marker)

site_map

**Questions to answer from the map**
* Are all launch sites in proximity to the Equator line?
* Are all launch sites in very close proximity to the coast?

*Your findings:* (write them here)

## Task 2: Mark the success/failed launches for each site

In [7]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


In [8]:
marker_cluster = MarkerCluster()

# class = 1 -> green, class = 0 -> red
spacex_df['marker_color'] = spacex_df['class'].apply(lambda x: 'green' if x == 1 else 'red')
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


In [9]:
# Add marker_cluster to the current site_map
site_map.add_child(marker_cluster)

for index, record in spacex_df.iterrows():
    marker = folium.Marker(
        [record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color'])
    )
    marker_cluster.add_child(marker)

site_map

From the colour-labelled markers you can see which launch sites have relatively high success rates.

## Task 3: Calculate the distances between a launch site and its proximities

In [10]:
# MousePosition shows the Lat/Long under the cursor, so you can read off coordinates
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

In [11]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    return R * c

### Coastline

In [12]:
# Launch site: CCAFS LC-40
launch_site_lat, launch_site_lon = 28.562302, -80.577356
# Closest coastline point (read from MousePosition)
coastline_lat, coastline_lon = 28.56367, -80.57163

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)
print(f"{distance_coastline:.2f} KM")

0.58 KM


In [13]:
coastline_coordinate = [coastline_lat, coastline_lon]

distance_marker = folium.Marker(
    coastline_coordinate,
    icon=DivIcon(
        icon_size=(20, 20),
        icon_anchor=(0, 0),
        html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
    )
)
site_map.add_child(distance_marker)

lines = folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon], coastline_coordinate],
    weight=1
)
site_map.add_child(lines)
site_map

### Railway, highway and city
The coordinates below are **approximate placeholders**. Replace them with the points you read off the map using MousePosition.

In [14]:
def add_proximity(site_map, site_lat, site_lon, point_lat, point_lon, label=''):
    distance = calculate_distance(site_lat, site_lon, point_lat, point_lon)
    folium.Marker(
        [point_lat, point_lon],
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12px; color:#d35400;"><b>%s %.2f KM</b></div>' % (label, distance),
        )
    ).add_to(site_map)
    folium.PolyLine(
        locations=[[site_lat, site_lon], [point_lat, point_lon]],
        weight=1
    ).add_to(site_map)
    return distance

# Replace these with coordinates you measure yourself
RAIL_LAT, RAIL_LON = 28.57206, -80.58527   # nearest railway (approx.)
HWY_LAT,  HWY_LON  = 28.56334, -80.57079   # nearest highway (approx.)
CITY_LAT, CITY_LON = 28.0836,  -80.6081    # Melbourne, FL (approx.)

railway_dist = add_proximity(site_map, launch_site_lat, launch_site_lon, RAIL_LAT, RAIL_LON, 'Railway')
highway_dist = add_proximity(site_map, launch_site_lat, launch_site_lon, HWY_LAT, HWY_LON, 'Highway')
city_dist    = add_proximity(site_map, launch_site_lat, launch_site_lon, CITY_LAT, CITY_LON, 'City')

print(f"Railway: {railway_dist:.2f} km | Highway: {highway_dist:.2f} km | City: {city_dist:.2f} km")
site_map

Railway: 1.33 km | Highway: 0.65 km | City: 53.33 km


## Findings
* Are launch sites in close proximity to railways?
* Are launch sites in close proximity to highways?
* Are launch sites in close proximity to the coastline?
* Do launch sites keep a certain distance away from cities?

*Typical explanation:* sites sit close to the coast (launching over water keeps debris away from people), close to rail and highways (moving heavy equipment), and far from cities (safety buffer).

## Next Steps
Build a dashboard using Plotly Dash on the detailed launch records.